# ATOMCHANGER: VRG

#### 1. find rxn site
#### 2. change halogens (to other halogens or CF3...), oxygens (to other halogens(for OH) or R2NMe)
#### &nbsp;&nbsp;nitrogens (primary>OH, secondary>OR, tertiary>PR2), carbons (far from O or N > ROR, RNR(Me))
#### &nbsp;&nbsp;swap atoms in functional groups

In [6]:
nrows = 50000
similarity_value = 0.85 # for clustering
n_iter=7 # for atom changer

In [7]:
#file load
import pandas as pd

dataset_train = pd.read_csv('original_datasets/raw_train.csv', nrows=nrows)
dataset_val = pd.read_csv('original_datasets/raw_val.csv', nrows=int(nrows*0.2))

def split_reaction(smiles_reaction):
    inputs, output = smiles_reaction.split('>>')  # split with '>>'        
    return pd.Series([inputs, output])

dataset_train[['inputs', 'output']] = dataset_train.iloc[:, 2].apply(split_reaction)
dataset_val[['inputs', 'output']] = dataset_val.iloc[:, 2].apply(split_reaction)
smiles_columns = ['inputs', 'output']

print(dataset_val) #check

                   id class  \
0        US08329716B2   UNK   
1          US06051718   UNK   
2        US07504410B2   UNK   
3          US04960769   UNK   
4     US20110092505A1   UNK   
...               ...   ...   
4996  US20140194411A1   UNK   
4997  US20090149445A1   UNK   
4998     US08710243B2   UNK   
4999  US20130303532A1   UNK   
5000     US06518265B1   UNK   

                          reactants>reagents>production  \
0     O=C(O[C:1](=[O:2])[C:3]([F:4])([F:5])[F:6])C(F...   
1     CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:...   
2     O=C(O[C:1](=[O:2])[C:3]([F:4])([F:5])[F:6])C(F...   
3     CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:...   
4     CC(C)(C)OC(=O)O[C:6]([O:5][C:2]([CH3:1])([CH3:...   
...                                                 ...   
4996  O[C:1]1([CH:2]2[CH2:3][CH2:4]2)[c:5]2[c:6]([cH...   
4997  Br[c:1]1[cH:2][cH:3][cH:4][c:5]([CH:6]=[C:7]2[...   
4998  O=[CH:1][n:2]1[c:3](-[c:4]2[c:5]([CH3:6])[n:7]...   
4999  Br[c:1]1[n:2][c:3]([NH:4][CH2:5

In [8]:
#reaction_site
import augment_models.reaction_site as reaction_site

dataset_train[['class', 'fg_site']] = pd.DataFrame(
    dataset_train.apply(
        lambda row: reaction_site.get_reaction_center(row['inputs'], row['output'], depth=1)[:2], axis=1).tolist())
dataset_val[['class', 'fg_site']] = pd.DataFrame(
    dataset_val.apply(
        lambda row: reaction_site.get_reaction_center(row['inputs'], row['output'], depth=1)[:2], axis=1).tolist())

In [9]:
#clustering_fg
import re
from collections import defaultdict
from rdkit import Chem
from rdkit import rdBase
import augment_models.functionalizer_synt as functionalizer_synt

rdBase.DisableLog('rdApp.*')

def remove_atom_map(smiles):
    return re.sub(r":\d+", "", smiles)

def get_mol(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except:
        return None
        
def get_cluster_key(smiles):
    mol = get_mol(smiles)
    if mol:
        return Chem.MolToSmiles(mol, canonical=True)
    return None

reaction_groups = defaultdict(list)
reaction_groups_val = defaultdict(list)

for idx, row in dataset_train.iterrows():
    fg_raw = row['fg_site']
    if not fg_raw:
        continue

    fg_clean = remove_atom_map(fg_raw)
    key = get_cluster_key(fg_clean)
    assigned_key = None

    if key:
        is_assigned = False
        for key_old, group_rows in reaction_groups.items():
            existing_fg_clean = remove_atom_map(group_rows[0]['fg_site'])
            sim = functionalizer_synt.calculate_similarity(existing_fg_clean, fg_clean, 2)
            if sim >= similarity_value:
                reaction_groups[key_old].append(row)
                is_assigned = True
                break

        if not is_assigned:
            reaction_groups[key].append(row)

print(f"clusters (train): {len(reaction_groups)}")

for idx, row in dataset_val.iterrows():
    fg_raw = row['fg_site']
    if not fg_raw:
        continue

    fg_clean = remove_atom_map(fg_raw)
    key = get_cluster_key(fg_clean)

    if key:
        is_assigned = False
        for key_old, group_rows in reaction_groups_val.items():
            existing_fg_clean = remove_atom_map(group_rows[0]['fg_site'])
            sim = functionalizer_synt.calculate_similarity(existing_fg_clean, fg_clean, 2)
            if sim >= similarity_value:
                reaction_groups_val[key_old].append(row)
                is_assigned = True
                break

        if not is_assigned:
            reaction_groups_val[key].append(row)

print(f"clusters (val): {len(reaction_groups_val)}")

## this can be used further cluster-based changer system

clusters (train): 1961
clusters (val): 487


In [10]:
import csv
import augment_models.atom_changer_synt as atom_changer_synt

def chunk_dict(d, chunk_size):
    items = list(d.items())
    for i in range(0, len(items), chunk_size):
        yield dict(items[i:i+chunk_size])

def flatten_result_to_rows(results):
    rows = []
    for idx, entry in enumerate(results):
        generated_inputs = entry.get('generated_input', [])
        generated_outputs = entry.get('generated_output', [])

        for gi, go in zip(generated_inputs, generated_outputs):
            rows.append({
                'inputs': gi,
                'output': go,
                'reaction_center': entry['cluster'],
                'source_row': entry['id']
            })
    return rows

def remove_same_data(data):
    seen = set()
    unique_data = []
    for entry in data:
        tup = tuple((k, str(entry[k])) for k in sorted(entry))
        if tup not in seen:
            seen.add(tup)
            unique_data.append(entry)
    return unique_data

def append_csv_chunk(data, filename, write_header=False):
    mode = 'w' if write_header else 'a'
    with open(filename, mode, newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['source_row', 'reaction_center', 'input>>output'])
        if write_header:
            writer.writeheader()
        
        formatted_data = []
        for row in data:
            new_row = {
                'source_row': row['source_row'],
                'reaction_center': row['reaction_center'],
                'input>>output': f"{row['inputs']}>>{row['output']}"
            }
            formatted_data.append(new_row)

        writer.writerows(formatted_data)
    print(f"[Appended] {len(data)} rows → {filename}")

# ---------- Training ----------
train_csv_path = 'augmented_datasets/augmented_atomchanger_train.csv'
print("[Start] AtomChanger augmentation for training...")

for i, chunk in enumerate(chunk_dict(reaction_groups, 25)):
    print(f"[Train] Chunk {i + 1}")
    results = atom_changer_synt.atom_changer_process(chunk, n_iter=n_iter)
    if results:
        flattened = flatten_result_to_rows(results)
        chunk_result = remove_same_data(flattened)
        append_csv_chunk(chunk_result, train_csv_path, write_header=(i == 0))

# ---------- Validation ----------
val_csv_path = 'augmented_datasets/augmented_atomchanger_val.csv'
print("[Start] AtomChanger augmentation for validation...")

for i, chunk in enumerate(chunk_dict(reaction_groups_val, 25)):
    print(f"[Val] Chunk {i + 1}")
    results = atom_changer_synt.atom_changer_process(chunk, n_iter=n_iter)
    if results:
        flattened = flatten_result_to_rows(results)
        chunk_result = remove_same_data(flattened)
        append_csv_chunk(chunk_result, val_csv_path, write_header=(i == 0))

[Start] AtomChanger augmentation for training...
[Train] Chunk 1
[Appended] 3718 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 2
[Appended] 1063 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 3
[Appended] 1129 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 4
[Appended] 817 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 5
[Appended] 651 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 6
[Appended] 519 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 7
[Appended] 474 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 8
[Appended] 492 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 9
[Appended] 524 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 10
[Appended] 415 rows → augmented_datasets/augmented_atomchanger_train.csv
[Train] Chunk 11
[Appended] 205 rows → augmented_datasets